# Módulo 51: Análisis de regresión

Autor: Juliho Castillo Colmenares

> Lección complementaria del ejercicio [m51-analisis-de-regresion.ipynb](m51-analisis-de-regresion.ipynb). No es una tarea entregada del programa.

La **regresión lineal múltiple** explica una variable numérica a partir de varias otras. Es el punto de partida de casi todo el modelado estadístico: sus coeficientes se interpretan, sus supuestos se pueden probar y su solución se escribe con una sola fórmula. En esta lección se deduce esa fórmula, se ve cuándo falla y se mide la calidad del ajuste:

1. El modelo $y = X\beta + \varepsilon$
2. Mínimos cuadrados y la ecuación normal
3. La columna de intercepto
4. Cribado por correlación y sus límites
5. Multicolinealidad perfecta
6. Métricas: $R^2$, MSE y MAE
7. Evaluación dentro y fuera de muestra
8. Errores frecuentes
9. Ejercicio

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

Se usa `kc-house-data.csv`, con 21.613 ventas de casas en el condado de King (Washington). Para que los cálculos sean rápidos y legibles se toma una **submuestra aleatoria de 2.000 casas**.

In [2]:
casas_completo = pd.read_csv("kc-house-data.csv")
casas = casas_completo.sample(n=2000, random_state=42).reset_index(drop=True)
print(casas_completo.shape, "->", casas.shape)

columnas = ["price", "sqft_living", "sqft_above", "sqft_basement", "bedrooms",
            "bathrooms", "grade", "view", "yr_built"]
casas[columnas].describe().round(1).T[["mean", "std", "min", "max"]]

(21613, 21) -> (2000, 21)


,mean,std,min,max
price,550823.5,384508.6,82500.0,5350000.0
sqft_living,2087.8,914.9,520.0,8000.0
sqft_above,1773.8,811.9,480.0,6720.0
sqft_basement,314.0,455.9,0.0,2330.0
bedrooms,3.4,0.9,0.0,8.0
bathrooms,2.1,0.8,0.0,6.0
grade,7.7,1.2,4.0,13.0
view,0.2,0.8,0.0,4.0
yr_built,1970.1,29.3,1900.0,2015.0


## 1. El modelo $y = X\beta + \varepsilon$

Con $n$ observaciones y $p$ predictores, el modelo lineal se escribe en forma matricial:

$$y = X\beta + \varepsilon, \qquad y_i = \beta_0 + \beta_1 x_{i1} + \dots + \beta_p x_{ip} + \varepsilon_i.$$

| Símbolo | Tamaño | Significado |
| --- | --- | --- |
| $y$ | $n \times 1$ | Variable respuesta (precio) |
| $X$ | $n \times (p+1)$ | Matriz de diseño: una columna de unos y una por predictor |
| $\beta$ | $(p+1) \times 1$ | Coeficientes desconocidos |
| $\varepsilon$ | $n \times 1$ | Errores: lo que el modelo no explica |

Cada $\beta_j$ es el cambio esperado en $y$ cuando $x_j$ sube una unidad y los demás predictores se mantienen fijos.

In [3]:
predictores = ["sqft_living", "bedrooms", "grade"]
X = np.column_stack([np.ones(len(casas)), casas[predictores]])   # columna de unos + predictores
y = casas["price"].to_numpy()
print("X:", X.shape, " y:", y.shape)
print(X[:3])

X: (2000, 4)  y: (2000,)
[[1.00e+00 2.07e+03 4.00e+00 8.00e+00]
 [1.00e+00 2.90e+03 5.00e+00 8.00e+00]
 [1.00e+00 3.77e+03 4.00e+00 1.10e+01]]


## 2. Mínimos cuadrados y la ecuación normal

Se eligen los coeficientes que minimizan la suma de residuos al cuadrado:

$$S(\beta) = \lVert y - X\beta \rVert^2 = y^\top y - 2\beta^\top X^\top y + \beta^\top X^\top X \beta.$$

Igualando a cero el gradiente, $\nabla S(\beta) = -2X^\top y + 2X^\top X\beta = 0$, se obtienen las **ecuaciones normales** $X^\top X \beta = X^\top y$. Si $X^\top X$ es invertible, que ocurre cuando las columnas de $X$ son linealmente independientes, la solución es única:

$$\hat\beta = (X^\top X)^{-1} X^\top y.$$

En la práctica no se calcula la inversa: se resuelve el sistema con `np.linalg.solve`, que es más estable.

In [4]:
beta = np.linalg.solve(X.T @ X, X.T @ y)
print(pd.Series(beta, index=["intercepto"] + predictores).round(2))

# La misma solución con scikit-learn
lr = LinearRegression().fit(casas[predictores], y)
print("sklearn:", np.round(np.r_[lr.intercept_, lr.coef_], 2))

intercepto    -582994.35
sqft_living       204.19
bedrooms       -35227.85
grade          107911.29
dtype: float64
sklearn: [-5.8299435e+05  2.0419000e+02 -3.5227850e+04  1.0791129e+05]


Manteniendo fijos los dormitorios y la calidad (`grade`), cada pie cuadrado adicional suma en promedio unos 204 dólares al precio. El coeficiente negativo de `bedrooms` no dice que un dormitorio reste valor a cualquier casa: compara casas **de la misma superficie**, en las que más dormitorios significa habitaciones más pequeñas.

## 3. La columna de intercepto

La columna de unos estima $\beta_0$, el valor esperado de $y$ cuando todos los predictores valen 0. Sin ella, el modelo queda obligado a pasar por el origen, y los demás coeficientes se distorsionan para compensar.

In [5]:
sin_intercepto = LinearRegression(fit_intercept=False).fit(casas[predictores], y)
print("Coeficientes sin intercepto:", sin_intercepto.coef_.round(2))
print(f"R² con intercepto: {lr.score(casas[predictores], y):.3f}")
print(f"R² sin intercepto: {r2_score(y, sin_intercepto.predict(casas[predictores])):.3f}")

Coeficientes sin intercepto: [   288.38 -77516.47  28449.81]
R² con intercepto: 0.530
R² sin intercepto: 0.500


## 4. Cribado por correlación y sus límites

Una forma rápida de preseleccionar predictores es calcular su correlación con la respuesta y quedarse con los que superan un umbral. Es útil para explorar, pero tiene límites:

- Solo mide relaciones **lineales y marginales**: una variable sin correlación por sí sola puede ser útil junto con otras.
- No detecta **redundancia**: dos predictores casi idénticos pasan el filtro los dos.
- Si se calcula con todos los datos, la selección ya "vio" el conjunto de prueba.

In [6]:
corr_precio = casas[columnas].corr()["price"].drop("price").sort_values(ascending=False)
print(corr_precio.round(3))

print("\nCorrelación entre predictores que pasan el filtro:")
print(casas[["sqft_living", "sqft_above", "grade"]].corr().round(3))

sqft_living      0.689
grade            0.671
sqft_above       0.595
bathrooms        0.491
view             0.421
bedrooms         0.324
sqft_basement    0.324
yr_built         0.056
Name: price, dtype: float64

Correlación entre predictores que pasan el filtro:
             sqft_living  sqft_above  grade
sqft_living        1.000       0.867  0.763
sqft_above         0.867       1.000  0.750
grade              0.763       0.750  1.000


`sqft_living`, `sqft_above` y `grade` superan con holgura cualquier umbral habitual, pero están muy correlacionadas entre sí: aportan en buena parte la misma información.

## 5. Multicolinealidad perfecta

Si una columna de $X$ es combinación lineal exacta de otras, $X$ no tiene rango completo y $X^\top X$ es **singular**: no tiene inversa y las ecuaciones normales tienen infinitas soluciones. En estos datos ocurre con la superficie, porque `sqft_living = sqft_above + sqft_basement` en todas las filas.

Para diagnosticarlo:

| Herramienta | Señal de problema |
| --- | --- |
| `np.linalg.matrix_rank(X)` | Menor que el número de columnas |
| `np.linalg.cond(X)` | **Número de condición** enorme: $\kappa = \sigma_{\max}/\sigma_{\min}$ |

In [7]:
print("¿sqft_living = sqft_above + sqft_basement?",
      (casas["sqft_living"] == casas["sqft_above"] + casas["sqft_basement"]).all())

cols_col = ["sqft_living", "sqft_above", "sqft_basement", "grade"]
X_col = np.column_stack([np.ones(len(casas)), casas[cols_col]])
print("Columnas:", X_col.shape[1], " rango:", np.linalg.matrix_rank(X_col))
print(f"Número de condición de X:     {np.linalg.cond(X_col):.2e}")
print(f"Número de condición sin la redundante: "
      f"{np.linalg.cond(np.column_stack([np.ones(len(casas)), casas[['sqft_living', 'sqft_above', 'grade']]])):.2e}")

¿sqft_living = sqft_above + sqft_basement? True
Columnas: 5  rango: 4
Número de condición de X:     3.12e+15
Número de condición sin la redundante: 2.40e+04


Hay dos salidas correctas:

1. **Eliminar la columna redundante.** La información no se pierde, porque se puede reconstruir con las otras.
2. **Usar `np.linalg.lstsq` o `np.linalg.pinv`**, que devuelven la solución de norma mínima. Las predicciones son válidas, pero los coeficientes individuales no son únicos y no deben interpretarse.

In [8]:
beta_lstsq, *_ = np.linalg.lstsq(X_col, y, rcond=None)
X_sin = np.column_stack([np.ones(len(casas)), casas[["sqft_living", "sqft_above", "grade"]]])
beta_sin = np.linalg.solve(X_sin.T @ X_sin, X_sin.T @ y)

print("Coeficientes lstsq (norma mínima):", beta_lstsq.round(2))
print("Coeficientes sin sqft_basement:   ", beta_sin.round(2))
print("¿Mismas predicciones?", np.allclose(X_col @ beta_lstsq, X_sin @ beta_sin))

Coeficientes lstsq (norma mínima): [-7.3315017e+05  1.2463000e+02  2.7840000e+01  9.6800000e+01
  1.2326784e+05]
Coeficientes sin sqft_basement:    [-7.3315017e+05  2.2143000e+02 -6.8960000e+01  1.2326784e+05]
¿Mismas predicciones? True


Los coeficientes cambian, pero las predicciones son idénticas: con colinealidad perfecta hay muchas combinaciones de coeficientes que producen el mismo ajuste.

## 6. Métricas: $R^2$, MSE y MAE

Con $\hat y_i$ la predicción y $\bar y$ la media de la respuesta:

$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum_i (y_i - \hat y_i)^2}{\sum_i (y_i - \bar y)^2}, \qquad \text{MSE} = \frac1n \sum_i (y_i - \hat y_i)^2, \qquad \text{MAE} = \frac1n \sum_i \lvert y_i - \hat y_i\rvert.$$

| Métrica | Lectura |
| --- | --- |
| $R^2$ | Proporción de la variabilidad de $y$ explicada. 1 es perfecto; 0 equivale a predecir siempre la media; puede ser negativo |
| MSE | Error cuadrático medio, en unidades de $y$ al cuadrado; castiga más los errores grandes |
| MAE | Error absoluto medio, en las unidades de $y$; fácil de comunicar |

In [9]:
y_hat = X @ beta
ss_res = np.sum((y - y_hat) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)

print(f"R² a mano: {1 - ss_res / ss_tot:.4f}   sklearn: {r2_score(y, y_hat):.4f}")
print(f"MSE: {mean_squared_error(y, y_hat):.4e}   RMSE: {np.sqrt(mean_squared_error(y, y_hat)):,.0f}")
print(f"MAE: {mean_absolute_error(y, y_hat):,.0f}")

R² a mano: 0.5295   sklearn: 0.5295
MSE: 6.9527e+10   RMSE: 263,679
MAE: 171,264


## 7. Evaluación dentro y fuera de muestra

El $R^2$ calculado con los mismos datos del ajuste (**dentro de muestra**) es optimista: el modelo eligió sus coeficientes para esos datos. Para saber cómo predice casas nuevas se separa un **conjunto de prueba** que no participa en el ajuste.

Añadir predictores siempre sube el $R^2$ de entrenamiento, aunque sean ruido puro. El de prueba, en cambio, puede bajar.

In [10]:
rng = np.random.default_rng(42)
base = casas[["sqft_living", "grade", "view", "bathrooms", "yr_built"]]
ruido = pd.DataFrame(rng.normal(size=(len(casas), 300)), columns=[f"ruido_{i}" for i in range(300)])

for nombre, datos in [("5 predictores", base), ("5 + 300 de ruido", pd.concat([base, ruido], axis=1))]:
    X_tr, X_te, y_tr, y_te = train_test_split(datos, y, test_size=0.3, random_state=42)
    m = LinearRegression().fit(X_tr, y_tr)
    print(f"{nombre:>17}: R² entrenamiento={m.score(X_tr, y_tr):.3f}, R² prueba={m.score(X_te, y_te):.3f}")

    5 predictores: R² entrenamiento=0.608, R² prueba=0.619
 5 + 300 de ruido: R² entrenamiento=0.692, R² prueba=0.496


## 8. Errores frecuentes

**Invertir $X^\top X$ cuando es singular.** Con columnas colineales, `np.linalg.inv` puede lanzar `LinAlgError` o, por el redondeo, devolver una "inversa" con números enormes y coeficientes sin sentido. Ninguna de las dos señales es fiable: antes de invertir, hay que revisar el rango y el número de condición.

In [11]:
try:
    beta_inv = np.linalg.inv(X_col.T @ X_col) @ X_col.T @ y
    print("inv no falló; coeficientes:", beta_inv.round(1))
    print(f"R² con esos coeficientes: {r2_score(y, X_col @ beta_inv):.3f}")
except np.linalg.LinAlgError as error:
    print("LinAlgError:", error)

# Forma correcta: quitar la columna redundante o usar lstsq
print(f"R² con lstsq: {r2_score(y, X_col @ beta_lstsq):.3f}")

inv no falló; coeficientes: [-7.3032739e+06  5.8760000e+02  2.0034000e+03  4.8120000e+02
  1.2326780e+05]
R² con esos coeficientes: -53.068
R² con lstsq: 0.530


**Reportar solo el $R^2$ de entrenamiento.** Como se vio en la sección 7, puede subir aunque el modelo prediga peor. La evaluación se hace con datos que el modelo no ha visto.

**Leer un $R^2$ negativo como un error de cálculo.** Fuera de muestra, o con coeficientes mal estimados, un $R^2$ negativo es posible y significa que el modelo predice peor que la media:

In [12]:
prediccion_constante_mala = np.full_like(y, y.mean() * 1.5)
print(f"R² de predecir 1.5 veces la media: {r2_score(y, prediccion_constante_mala):.3f}")

R² de predecir 1.5 veces la media: -0.513


**Olvidar la columna de unos** al construir $X$ a mano: el modelo pasa por el origen sin avisar.

## 9. Ejercicio

Una empresa registra el consumo eléctrico mensual (kWh) de 500 edificios. Simula los datos así:

```python
rng = np.random.default_rng(42)
n = 500
oficinas = rng.uniform(200, 2000, n)        # m² de oficinas
bodega = rng.uniform(0, 800, n)             # m² de bodega
empleados = rng.integers(5, 150, n)
edificios = pd.DataFrame({"oficinas": oficinas, "bodega": bodega,
                          "area_total": oficinas + bodega, "empleados": empleados})
edificios["kwh"] = 500 + 3.0 * oficinas + 1.2 * bodega + 25 * empleados + rng.normal(0, 600, n)
```

1. Construye $X$ con intercepto y las cuatro variables, y calcula su rango y su número de condición.
2. Explica por qué $X^\top X$ es singular.
3. Divide en entrenamiento (70 %) y prueba (30 %) y ajusta el modelo de dos formas: con `np.linalg.lstsq` y eliminando `area_total`.
4. Comprueba que las predicciones de prueba coinciden y calcula $R^2$, MSE y MAE dentro y fuera de muestra.
5. Interpreta el coeficiente de `oficinas` en el modelo sin `area_total`.

### Solución

In [13]:
rng = np.random.default_rng(42)
n = 500
oficinas = rng.uniform(200, 2000, n)
bodega = rng.uniform(0, 800, n)
empleados = rng.integers(5, 150, n)
edificios = pd.DataFrame({"oficinas": oficinas, "bodega": bodega,
                          "area_total": oficinas + bodega, "empleados": empleados})
edificios["kwh"] = 500 + 3.0 * oficinas + 1.2 * bodega + 25 * empleados + rng.normal(0, 600, n)

# 1. Diagnóstico
todas = ["oficinas", "bodega", "area_total", "empleados"]
Xe = np.column_stack([np.ones(n), edificios[todas]])
print("Columnas:", Xe.shape[1], " rango:", np.linalg.matrix_rank(Xe), f" cond: {np.linalg.cond(Xe):.2e}")
# 2. area_total = oficinas + bodega: una columna es suma de otras dos

Columnas: 5  rango: 4  cond: 1.13e+16


In [14]:
# 3. División y dos ajustes
idx_tr, idx_te = train_test_split(np.arange(n), test_size=0.3, random_state=42)
ye = edificios["kwh"].to_numpy()

b_lstsq, *_ = np.linalg.lstsq(Xe[idx_tr], ye[idx_tr], rcond=None)

sin_total = ["oficinas", "bodega", "empleados"]
Xr = np.column_stack([np.ones(n), edificios[sin_total]])
b_red = np.linalg.solve(Xr[idx_tr].T @ Xr[idx_tr], Xr[idx_tr].T @ ye[idx_tr])

# 4. Mismas predicciones y métricas
print("Predicciones de prueba iguales:", np.allclose(Xe[idx_te] @ b_lstsq, Xr[idx_te] @ b_red))
for nombre, idx in [("entrenamiento", idx_tr), ("prueba", idx_te)]:
    pred = Xr[idx] @ b_red
    print(f"{nombre:>13}: R²={r2_score(ye[idx], pred):.3f}, "
          f"MSE={mean_squared_error(ye[idx], pred):,.0f}, MAE={mean_absolute_error(ye[idx], pred):,.0f}")

# 5. Coeficientes del modelo reducido
print(pd.Series(b_red, index=["intercepto"] + sin_total).round(2))

Predicciones de prueba iguales: True
entrenamiento: R²=0.907, MSE=368,171, MAE=476
       prueba: R²=0.872, MSE=474,345, MAE=568
intercepto    397.59
oficinas        3.11
bodega          1.06
empleados      24.68
dtype: float64


Con la bodega y los empleados fijos, cada metro cuadrado adicional de oficinas suma unos 3 kWh al mes, cerca del valor 3.0 usado en la simulación. El $R^2$ de prueba (0.872) queda cerca del de entrenamiento (0.907), así que no hay señales de sobreajuste.

## Resumen

- El modelo lineal se escribe $y = X\beta + \varepsilon$, con una columna de unos en $X$ para el intercepto.
- Minimizar $\lVert y - X\beta\rVert^2$ lleva a las ecuaciones normales y a $\hat\beta = (X^\top X)^{-1}X^\top y$ cuando $X$ tiene rango completo.
- Una columna que es combinación exacta de otras hace singular a $X^\top X$; se detecta con el rango y el número de condición, y se resuelve quitando la columna o con `lstsq`/`pinv`.
- El cribado por correlación es exploratorio: ignora la redundancia entre predictores.
- $R^2$, MSE y MAE resumen el ajuste, y deben medirse también fuera de muestra.

En el **Módulo 52** se estudian las pruebas de hipótesis sobre los coeficientes: cuándo un $\hat\beta_j$ es distinto de cero de forma significativa.